# Visualize results

This is a notebook to visualize the results of different metric calculations for different datasets and conditions. We want to see:

- For each model, how do the scores change as we go further into the model (increasing relative depth)?
Different figures for a) each model b) each dataset c) all controls, or just trained plm + probe + embedding controls d) mean scores, or elementwise scores
X axis: relative depth
Y axis: metric of choice
Legend: embedding controls

- Do different models have different patterns in how their scores change through the layers, for a given dataset?


In [ ]:
%load_ext autoreload
%autoreload 2

### Imports

In [ ]:
import warnings

# This will ignore all UserWarning messages
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
import polars as pl
import json
import seaborn as sns
import matplotlib.pyplot as plt
from functools import reduce

from project.utils.strs import linear_probe_metrics_dir, linear_probe_compiled_results_dir, linear_probe_results_dir, linear_probe_figures_dir
from project.utils.functions import load_config

In [ ]:
# Make dirs if they don't exist already
for directory in [linear_probe_metrics_dir, linear_probe_compiled_results_dir, linear_probe_results_dir, linear_probe_figures_dir]:
    directory.mkdir(parents=True, exist_ok=True)

### Mapping dicts

In [ ]:
datasets = [
    # "protein properties"
	"prot_param",
    # Dependent largely on linear sequence
	"interpro_conserved_site",
	"interpro_repeat",
    # "uniprot_peptide",
    # "uniprot_post_translational_modification",
    # "uniprot_phosphorylation",
    # "uniprot_lipidation",
	"biomap_localization_prediction",
    # # Secondary structure
    # "biomap_ssp_q3",
    # "biomap_ssp_q8",
	# "uniprot_secondary_structure",
    # # Dependent largely on tertiary structure
    "uniprot_functional_sites",
	"interpro_binding_site",
	# "biomap_metal_ion_binding",
	"interpro_active_site",
    "interpro_domain",
    "interpro_family",
    # "interpro_homologous_superfamily",
    # # Dependent on cellular context
    # "uniprot_topology",
    "GO_cc",
    "GO_mf",
    "GO_bp",
]

protein_feature_colormapping = {
    # Baseline 1D (Grey)
    "prot_param": "#708090",                       # SlateGray (Baseline)

    # 1D: Sequence Signals (Orange Flow)
    "uniprot_peptide": "#FFB88C",                   # Light Orange
    "interpro_conserved_site": "#FF8C00",           # DarkOrange
    "biomap_localization_prediction": "#E65100",    # Deep Burnt Orange

    # 2D: Secondary Structure (Blue Flow)
    "interpro_repeat": "#A2D2FF",                   # Light Sky Blue
    "uniprot_secondary_structure": "#5D9CEC",       # Soft Blue
    "biomap_ssp_q3": "#3498DB",                     # Bright Blue
    "biomap_ssp_q8": "#1E3A8A",                     # Deep Royal Blue (Flowing toward 3D)
    
    # 3D: Pure Structural Architecture (Light Green Flow)
    "interpro_homologous_superfamily": "#D1FAE5",  # Mint Cream
    "interpro_family": "#A7F3D0",                   # Pale Emerald
    "interpro_domain": "#6EE7B7",                   # Light Sea Green
    "uniprot_topology": "#34D399",                  # Medium Emerald

    # 3D + System Hybrid: Interactive/Active (Dark Green Flow)
    "uniprot_functional_sites": "#065F46",          # Dark Emerald
    "interpro_binding_site": "#064E3B",             # Deep Hunter Green
    "biomap_metal_ion_binding": "#022C22",          # Near-Black Green
    "interpro_active_site": "#0F172A",              # Deepest Green-Grey
    "uniprot_post_translational_modification": "#1E3A1A", # Dark Forest
    "uniprot_phosphorylation": "#14532D",           # Rich Moss Green
    "uniprot_lipidation": "#064E3B",                # Deep Evergreen

    # 4D+: System-Level Biological Process (Magenta Flow)
    "GO_cc": "#F0ABFC",                             # Light Orchid
    "GO_mf": "#D946EF",                             # Steel Magenta
    "GO_bp": "#701A75",                             # Deep Plum/Grape
}

model_colormapping = {
    # ESM2 Models: Sky Blue Family
    'esm2_8m':    '#BFE5FF', # Very Pale Blue
    'esm2_35m':   '#82C0E9', # Sky Blue
    'esm2_150m':  '#1E78B4', # Steel Blue
    'esm2_650m':  '#004C8C', # Midnight Blue

    # Amplify Models: Orange/Vermillion Family
    'amplify_120m':  '#FFCC80', # Light Orange
    'amplify_350m':  '#E66101', # Burnt Orange/Vermillion

    # SAmplify Models: Green Family (High contrast against Blue/Orange)
    'samplify_120m': '#B2E2E2', # Light Mint
    'samplify_350m': '#006D2C', # Forest Green
}

model_rename_dict = {
    'amplify_120m': 'AMPLIFY 120M',
    'samplify_120m': 'SaAMPLIFY 120M',
    'amplify_350m': 'AMPLIFY 350M',
    'samplify_350m': 'SaAMPLIFY 350M',
    'esm2_8m': 'ESM2 8M',
    'esm2_35m': 'ESM2 35M',
    'esm2_150m': 'ESM2 150M',
    'esm2_650m': 'ESM2 650M'
}

dataset_rename_dict = {
    # Physical & Sequence Properties
    'prot_param': 'Protein Parameters (Physicochemical)',
    'uniprot_peptide': 'Peptide Signal Sequences',
    'biomap_localization_prediction': 'Subcellular Localization',
    
    # Secondary & Local Structure
    'biomap_ssp_q3': 'Secondary Structure (3-class)',
    'biomap_ssp_q8': 'Secondary Structure (8-class)',
    'uniprot_secondary_structure': 'Secondary Structure (UniProt)',
    'uniprot_topology': 'Transmembrane Topology',
    
    # Domains & Families
    'interpro_domain': 'InterPro Domains',
    'interpro_family': 'InterPro Families',
    'interpro_homologous_superfamily': 'Homologous Superfamilies',
    'interpro_repeat': 'Structural Repeats',
    
    # Sites & Modifications
    'interpro_conserved_site': 'Conserved Sites',
    'interpro_binding_site': 'Binding Sites',
    'interpro_active_site': 'Enzymatic Active Sites',
    'uniprot_functional_sites': 'Functional Sites',
    'biomap_metal_ion_binding': 'Metal Ion Binding',
    'uniprot_post_translational_modification': 'PTMs',
    'uniprot_phosphorylation': 'Phosphorylation Sites',
    'uniprot_lipidation': 'Lipidation Sites',
    
    # Gene Ontology
    'GO_cc': 'GO Cellular Component',
    'GO_mf': 'GO Molecular Function',
    'GO_bp': 'GO Biological Process',
}

### Reading in the data

In [ ]:
standard_cols = ['dataset',
 'model_name',
 'layer_num',
 'relative_depth',
 'plm_state',
 'plm_embeddings_normalized',
 'linear_probe_state',
 'control_type', 'fold']

element_dfs = []
for dataset in datasets:
    for d in linear_probe_metrics_dir.glob(f"*{dataset}*"):
        # Filter out older datasets we don't want
        if ('tiny' not in str(d)) and ('medium' not in str(d)) and ('512' not in str(d)):
            try:
                dataset_dfs = []
                for f in d.iterdir():
                    dataset_dfs.append(pl.read_parquet(f)) 
                # Combine
                df = pl.concat(dataset_dfs, how='diagonal_relaxed')
                # Only select columns that don't have elementwise metrics
                element_dfs.append(df.select(standard_cols + [c for c in df.columns if (('elementwise' in c) and (c not in standard_cols))]))

            except Exception as e:
                print(f"problem with {f}: {e}")

element_df = pl.concat(element_dfs, how='diagonal_relaxed')

# Adjust the dataset nomenclature and extract metadata
element_df = element_df.with_columns([
    # 1. Extract '512_cutoff' if present, otherwise default to 'standard' (or null)
    pl.col("dataset")
    .str.extract(r"_(512_cutoff)", 1)
    .fill_null("standard")
    .alias("data_subset"),

    # 2. Extract the split number
    pl.col("dataset")
    .str.extract(r"_split(\d+)$", 1)
    .fill_null("0")
    .alias("split"),

    # 3. Clean the dataset name by removing the split and the cutoff suffixes
    pl.col("dataset")
    .str.replace(r"_512_cutoff", "")  # Remove cutoff
    .str.replace(r"_split\d+$", "")   # Remove split
    .alias("dataset")
]).with_columns(
    pl.col("split").cast(pl.Int64)
)

element_df = element_df.filter(
    (pl.col('data_subset') == 'standard'),
    (pl.col('split') == 0))

In [ ]:
element_df

In [ ]:
# Uncomment to export the data
element_df.sort(by=['dataset', 'model_name', 'layer_num', 'plm_state', 'plm_embeddings_normalized', 'linear_probe_state', 'control_type']).write_parquet(linear_probe_compiled_results_dir / 'compiled_elementwise_results_all_probes_no_checkpoints.parquet.gz')

## Calculate Significant Differences

First, we want to identify which individual categories / classes in a dataset had significant differences between the different models or layers. There's no point in looking at a category where all layers and models have the same performance, or where the performance of the probes isn't significantly different from the controls.

In [ ]:
sig_prop_json = linear_probe_results_dir / f"elementwise_results_significant_categories.json"

if sig_prop_json.exists():
    with sig_prop_json.open('r') as f:
        significant_categories = json.load(f)
else:

    from scipy import stats
    from scikit_posthocs import posthoc_dunn
    import numpy as np

    significant_categories = {}
    metrics_to_plot = [
        'cosine_similarity_elementwise',
        'mse_elementwise',
        'pearson_corr_coef_elementwise',
        'r2_score_elementwise',
        'spearman_corr_coef_elementwise',
        'matthews_corrcoef_elementwise',
        'f1_score_elementwise',
        'accuracy_elementwise'
        ]
    min_significant_conditions = 3
    p_value_threshold = 0.005


    for dataset in datasets:
        # Load the original dataset
        try:
            dataset_config = load_config(dataset)
            dataset_df = pl.read_parquet(dataset_config['data_path'])
            label_cols = [col for col in dataset_df.columns if (dataset_df[col].dtype != pl.String)]

            dataset_filter = pl.col('dataset') == dataset

            filters = dataset_filter

            df_filtered = element_df.filter(filters).with_columns(
                pl.col('plm_state').replace_strict({'trained': 'trained PLM', 'un-trained': 'un-trained PLM'}), # renaming for label convenience
                pl.col('linear_probe_state').replace_strict({'trained': 'trained probe', 'naive': 'naive probe', 'un-trained': 'un-trained probe'}),
                pl.col('plm_embeddings_normalized').replace_strict({True: 'embeddings_normalized', False: 'embeddings_not_normalized'}),
                pl.col('control_type').replace_strict({'original': 'original sequence', 'scrambled': 'scrambled sequence', 'mean': 'mean embedding', 'random_gaussian': 'gaussian_embedding', 'l2_normalized': 'normalized_embedding'}),
                )

            # Define what constitutes a treatment (what we want to be significantly different than other states)
            treatment_conditions = (pl.col('plm_state') == 'trained PLM') & (pl.col('linear_probe_state') == 'trained probe') & (pl.col('control_type').is_in(['original sequence', 'normalized_embedding']))
            df_filtered = df_filtered.with_columns(
                pl.when(treatment_conditions)
                .then(pl.lit('Treatment'))
                .otherwise(pl.lit('Control'))
                .alias('control_group')
            )
            
            available_metrics = [col for col in df_filtered.columns if ((len(df_filtered.filter(~pl.col(col).is_null())[col]) > 0) and ('elementwise' in col))]

            sig_cat_metric = {}

            for metric in available_metrics:
                if (metric in available_metrics) and (metric in metrics_to_plot):
                    print(f"--- Analyzing Metric: {metric} ---")
                    
                    # Unpack the metric col
                    sub_df = df_filtered.select(standard_cols + ['control_group'] + [metric]).with_columns( # expand and rename columns
                        pl.col(metric).list.to_struct()
                        ).unnest(metric).rename(dict(zip([f'field_{i}' for i in range(len(label_cols))], label_cols)))        

                    df_to_unpivot = sub_df.select(standard_cols + ['control_group'] + label_cols).sort(['relative_depth', 'model_name', 'plm_state', 'plm_embeddings_normalized', 'linear_probe_state', 'control_type'])

                    # Use DataFrame.unpivot to get the long format
                    df_long = df_to_unpivot.unpivot(
                        index=standard_cols + ['control_group'],
                        on=label_cols,
                        variable_name='property',
                        value_name='score'
                    )

                    print(f"Applying statistical tests (p < {p_value_threshold})...")
                    if min_significant_conditions is not None:
                        print(f"Requiring at least {min_significant_conditions} pairwise differences for Model/Layer comparisons.")

                    # Convert to pandas for statistical testing with scipy
                    df_pd = df_long.to_pandas()

                    # Define the column for the Model/Layer group (for Kruskal-Wallis/Dunn's)
                    df_pd['model_layer_group'] = df_pd['model_name'].astype(str) + '_' + df_pd['relative_depth'].astype(str)

                    # Store properties that meet the significance criteria
                    significant_properties_A = set() # Original vs Control
                    significant_properties_B = set() # Model/Layer differences
                    
                    # --- Iterate over properties and apply tests ---
                    for prop, group_df in df_pd.groupby('property'):
                        
                        # Check that the treatment group is significantly different from the control group
                        original_scores = group_df[group_df['control_group'] == 'Treatment']['score'].values
                        control_scores = group_df[group_df['control_group'] == 'Control']['score'].values

                        # Ensure control group has enough data
                        if len(control_scores) >= 5 and len(np.unique(control_scores)) > 1:
                            # Two-sided Mann-Whitney U test
                            stat, p_val = stats.mannwhitneyu(original_scores, control_scores, alternative='two-sided')
                            # Add the 
                            if p_val < p_value_threshold:
                                significant_properties_A.add(prop)

                        treatment_df = group_df[group_df['control_group'] == 'Treatment'].copy()

                        # Re-group the Treatment data by model_layer_group to make sure we're only looking at differences between treatments
                        group_data = {name: data['score'].values for name, data in treatment_df.groupby('model_layer_group')}
                        group_names = list(group_data.keys())
                        
                        # Collect valid groups (at least 5 observations, not all identical)
                        valid_groups = []
                        valid_group_names = []
                        for name, g in group_data.items():
                            if len(g) >= 5 and len(np.unique(g)) > 1:
                                    valid_groups.append(g)
                                    valid_group_names.append(name)
                                    
                        if len(valid_groups) < 2:
                            continue # Not enough valid groups for Kruskal-Wallis
                        
                        # 1. Kruskal-Wallis Test
                        H, p_val_KW = stats.kruskal(*valid_groups)

                        if p_val_KW < p_value_threshold:
                            
                            # 2. Post-hoc Dunn's Test with Bonferroni correction
                            try:
                                # Perform Dunn's test using the property-specific DataFrame subset
                                p_values_posthoc = posthoc_dunn(
                                    a=group_df,
                                    val_col='score',
                                    group_col='model_layer_group',
                                    p_adjust='bonferroni'
                                )
                                
                                # Count how many pairwise comparisons are significant
                                # We use the lower triangle of the matrix (excluding the diagonal)
                                p_vals = p_values_posthoc.values
                                # Get a flattened list of p-values from the lower triangle (excluding diagonal)
                                lower_triangle_p_vals = p_vals[np.tril_indices(p_vals.shape[0], k=-1)]
                                
                                significant_comparisons_B = np.sum(lower_triangle_p_vals < p_value_threshold)

                                if significant_comparisons_B >= min_significant_conditions:
                                    significant_properties_B.add(prop)
                                    
                            except Exception as e:
                                # Handle cases where posthoc test fails (e.g., due to identical groups/groups with 0 variance not properly filtered)
                                print(f"Post-hoc test failed for property {prop}: {e}")


                    # --- Combine the results ---
                    final_significant_properties = sorted(list(significant_properties_A.intersection(significant_properties_B)))

                    if final_significant_properties:
                        sig_cat_metric[metric] = final_significant_properties

                    print(f"\n**Metric: {metric}**")
                    print("---")
                    if final_significant_properties:
                        print(f"Found **{len(final_significant_properties)}** statistically significant properties for metric **{metric}**:")
                        print(f"These properties meet both criteria (Treatment vs. Control difference AND Model/Layer difference of at least {min_significant_conditions} pairs):")
                        print(f"* {', '.join(final_significant_properties)}")
                    else:
                        print("No properties met both statistical significance criteria.")
                    print("---")
        except Exception as e:
            print(e)
            continue

        significant_categories[dataset] = sig_cat_metric

    with open(sig_prop_json, 'w') as f:
        json.dump(significant_categories, f)

## Plot All Significant Categories Per Dataset

After calculating significant differences, we plot all categories with significant differences for each dataset.

In [ ]:
# Configuration for plotting
plm_states = ['trained']
probe_states = ['trained']
model_order = ['amplify_120m', 'amplify_350m', 'samplify_120m', 'samplify_350m', 'esm2_150m', 'esm2_650m']
metric_to_plot = 'matthews_corrcoef_elementwise'  # Can be changed to other metrics
num_cols = 7
xaxis_var = 'layer_num'

# Helper function to extract model type and size
def get_model_info(model_name):
    parts = model_name.split('_')
    if 'esm2' in model_name:
        model_type = 'ESM2'
        size = int(parts[-1].replace('m', ''))
    elif 'amplify' in model_name:
        if 'samplify' in model_name: 
            model_type = 'SAMPLIFY'
        else:
            model_type = 'AMPLIFY'
        size = int(parts[-1].replace('m', ''))
    else:
        model_type = 'Other'
        size = 0
    return model_type, size

### Plotting **all** sig categories per dataset

Visual inspection of the curves: we're looking for cases where one class of model is different from the others (e.g. both ESM2 are higher than AMPLIFY models or vice versa, for a given set of layers)

In [ ]:
# Load significant categories from the calculation
sig_prop_json = linear_probe_results_dir / f"elementwise_results_significant_categories.json"

if sig_prop_json.exists():
    with sig_prop_json.open('r') as f:
        significant_categories = json.load(f)
else:
    print("Warning: Significant categories file not found. Please run the calculation cell first.")
    significant_categories = {}

# Plot all significant categories for each dataset
for dataset in datasets:
    try:
        if dataset not in significant_categories:
            print(f"Skipping {dataset}: no significant categories found")
            continue
        
        # Get significant categories for this dataset and metric
        if metric_to_plot not in significant_categories[dataset]:
            print(f"Skipping {dataset}: no significant categories for metric {metric_to_plot}")
            continue
        
        cats_to_show = significant_categories[dataset][metric_to_plot]
        
        if not cats_to_show:
            print(f"Skipping {dataset}: empty significant categories list")
            continue
        
        print(f"\nPlotting {dataset}: {len(cats_to_show)} significant categories")
        
        # Load original dataset (needed for label_cols)
        dataset_config = load_config(dataset)
        dataset_df = pl.read_parquet(dataset_config['data_path'])
        label_cols = [col for col in dataset_df.columns if (dataset_df[col].dtype != pl.String)]

        all_filters = [
            (pl.col('plm_embeddings_normalized') == False), # normalization filter
            (pl.col('plm_state').is_in(plm_states)), # plm training filter
            pl.col('linear_probe_state').is_in(probe_states), # probe training filter
            (pl.col('model_name').is_in(model_order)), # model filter
            (pl.col('dataset') == dataset), # dataset filter
            ~((pl.col('plm_state') == 'un-trained') & (pl.col('linear_probe_state') == 'un-trained')), # untrained filter
        ]

        # Apply filters
        combined_filter = reduce(lambda a, b: a & b, all_filters)

        # Remap names for legibility
        df_filtered = element_df.filter(combined_filter).with_columns(
                    pl.col('plm_state').replace_strict({'trained': 'trained PLM', 'un-trained': 'un-trained PLM'}), # renaming for label convenience
                    pl.col('linear_probe_state').replace_strict({'trained': 'trained probe', 'naive': 'naive probe', 'un-trained': 'un-trained probe'}),
                    pl.col('plm_embeddings_normalized').replace_strict({True: 'embeddings_normalized', False: 'embeddings_not_normalized'}),
                    pl.col('control_type').replace_strict({'original': 'original sequence', 'scrambled': 'scrambled sequence', 'mean': 'mean embedding', 'random_gaussian': 'gaussian_embedding', 'l2_normalized': 'normalized_embedding'}),
            ).with_columns(
                    pl.concat_str(['plm_state', 'linear_probe_state', 'control_type'], separator= ' & ').alias('condition'), # concatenate plm, linear probe and control type
            )

        # Add new columns for model type and size
        df_filtered = df_filtered.with_columns(
            # Use map_elements for Python function calls
            pl.col('model_name').map_elements(
                lambda x: get_model_info(x)[0], return_dtype=pl.String
            ).alias('model_type'),
            pl.col('model_name').map_elements(
                lambda x: get_model_info(x)[1], return_dtype=pl.Int64
            ).alias('model_size_m'),
        ).with_columns(
            (pl.col('model_type') + '_' + pl.col('model_size_m').cast(pl.String)).alias('model_group_size')
        )

        # Unpack the metric col
        cols_to_select = standard_cols + ['model_type', 'model_size_m', 'model_group_size']
        sub_df = df_filtered.select(cols_to_select + [metric_to_plot]).with_columns( # expand and rename columns
            pl.col(metric_to_plot).list.to_struct()
            ).unnest(metric_to_plot).rename(dict(zip([f'field_{i}' for i in range(len(label_cols))], label_cols)))        

        df_to_unpivot = sub_df.select(cols_to_select + label_cols)

        df_long = df_to_unpivot.unpivot(
                            index=cols_to_select,
                            on=label_cols,
                            variable_name='property',
                            value_name='score'
                        )

        # Filter to only significant categories
        df_long = df_long.filter(pl.col('property').is_in(cats_to_show)).with_columns(
            pl.col('property').replace_strict({cats_to_show[i]: i for i in range(len(cats_to_show))}).alias('cat_order')
        ).sort(by='cat_order')
        
        # Format property names: split on "__" if present and wrap long text over multiple lines
        def format_property_name(prop_name, max_line_length=40):
            """Format property name to split IPR identifier from description and wrap long text."""
            def wrap_text(text, max_len):
                """Wrap text to multiple lines at word boundaries (underscores or spaces)."""
                if len(text) <= max_len:
                    return text
                lines = []
                current_line = ""
                # Split on underscores to preserve them, but also allow breaking at them
                # Replace underscores with a special marker, split, then restore
                parts = text.split('_')
                for i, part in enumerate(parts):
                    # Add underscore before part (except first)
                    separator = "_" if i > 0 else ""
                    test_line = current_line + separator + part if current_line else part
                    
                    if len(test_line) <= max_len:
                        current_line = test_line
                    else:
                        if current_line:
                            lines.append(current_line)
                        current_line = part
                if current_line:
                    lines.append(current_line)
                return "\n".join(lines) if len(lines) > 1 else text
            
            if '__' in prop_name:
                parts = prop_name.split('__', 1)
                ipr_part = parts[0]
                desc_part = parts[1]
                # Wrap the description part if it's long
                desc_wrapped = wrap_text(desc_part, max_line_length)
                return f"{ipr_part}\n{desc_wrapped}"
            else:
                # No "__" separator, just wrap the whole thing if it's long
                return wrap_text(prop_name, max_line_length)
        
        # Convert to pandas and format property names for plotting
        df_pd = df_long.to_pandas()
        df_pd['property_formatted'] = df_pd['property'].apply(format_property_name)
        
        # Adjust number of columns based on number of categories
        cols = min(num_cols, len(cats_to_show))

        col_by = 'property_formatted'
        color_by = 'model_name'

        fig = sns.relplot(data = df_pd, 
            x= xaxis_var, 
            y = 'score',
            col = col_by,
            col_wrap = cols,
            kind = 'line', 
            markers=True,
            errorbar = 'se',
            hue = color_by, 
            hue_order=model_order,
            height = 3,
            aspect = 1.25,
            palette = model_colormapping,
            facet_kws={'sharey': False, 'sharex': False},
            )

        fig.set_titles(col_template='{col_name}', row_template='{row_name}')
        fig.tight_layout()
        
        plt.show()

    except Exception as e:
        print(e)
        continue

Looking at individual prot_params

In [ ]:
metric_to_plot = 'pearson_corr_coef_elementwise'
num_cols = 6

# Load significant categories from the calculation
sig_prop_json = linear_probe_results_dir / f"elementwise_results_significant_categories.json"

if sig_prop_json.exists():
    with sig_prop_json.open('r') as f:
        significant_categories = json.load(f)
else:
    print("Warning: Significant categories file not found. Please run the calculation cell first.")
    significant_categories = {}

# Plot all significant categories for each dataset
for dataset in datasets:
    try:
        if dataset not in significant_categories:
            print(f"Skipping {dataset}: no significant categories found")
            continue
        
        # Get significant categories for this dataset and metric
        if metric_to_plot not in significant_categories[dataset]:
            print(f"Skipping {dataset}: no significant categories for metric {metric_to_plot}")
            continue
        
        cats_to_show = significant_categories[dataset][metric_to_plot]
        
        if not cats_to_show:
            print(f"Skipping {dataset}: empty significant categories list")
            continue
        
        print(f"\nPlotting {dataset}: {len(cats_to_show)} significant categories")
        
        # Load original dataset (needed for label_cols)
        dataset_config = load_config(dataset)
        dataset_df = pl.read_parquet(dataset_config['data_path'])
        label_cols = [col for col in dataset_df.columns if (dataset_df[col].dtype != pl.String)]

        all_filters = [
            (pl.col('plm_embeddings_normalized') == False), # normalization filter
            (pl.col('plm_state').is_in(plm_states)), # plm training filter
            pl.col('linear_probe_state').is_in(probe_states), # probe training filter
            (pl.col('model_name').is_in(model_order)), # model filter
            (pl.col('dataset') == dataset), # dataset filter
            ~((pl.col('plm_state') == 'un-trained') & (pl.col('linear_probe_state') == 'un-trained')), # untrained filter
        ]

        # Apply filters
        combined_filter = reduce(lambda a, b: a & b, all_filters)

        # Remap names for legibility
        df_filtered = element_df.filter(combined_filter).with_columns(
                    pl.col('plm_state').replace_strict({'trained': 'trained PLM', 'un-trained': 'un-trained PLM'}), # renaming for label convenience
                    pl.col('linear_probe_state').replace_strict({'trained': 'trained probe', 'naive': 'naive probe', 'un-trained': 'un-trained probe'}),
                    pl.col('plm_embeddings_normalized').replace_strict({True: 'embeddings_normalized', False: 'embeddings_not_normalized'}),
                    pl.col('control_type').replace_strict({'original': 'original sequence', 'scrambled': 'scrambled sequence', 'mean': 'mean embedding', 'random_gaussian': 'gaussian_embedding', 'l2_normalized': 'normalized_embedding'}),
            ).with_columns(
                    pl.concat_str(['plm_state', 'linear_probe_state', 'control_type'], separator= ' & ').alias('condition'), # concatenate plm, linear probe and control type
            )

        # Add new columns for model type and size
        df_filtered = df_filtered.with_columns(
            # Use map_elements for Python function calls
            pl.col('model_name').map_elements(
                lambda x: get_model_info(x)[0], return_dtype=pl.String
            ).alias('model_type'),
            pl.col('model_name').map_elements(
                lambda x: get_model_info(x)[1], return_dtype=pl.Int64
            ).alias('model_size_m'),
        ).with_columns(
            (pl.col('model_type') + '_' + pl.col('model_size_m').cast(pl.String)).alias('model_group_size')
        )

        # Unpack the metric col
        cols_to_select = standard_cols + ['model_type', 'model_size_m', 'model_group_size']
        sub_df = df_filtered.select(cols_to_select + [metric_to_plot]).with_columns( # expand and rename columns
            pl.col(metric_to_plot).list.to_struct()
            ).unnest(metric_to_plot).rename(dict(zip([f'field_{i}' for i in range(len(label_cols))], label_cols)))        

        df_to_unpivot = sub_df.select(cols_to_select + label_cols)

        df_long = df_to_unpivot.unpivot(
                            index=cols_to_select,
                            on=label_cols,
                            variable_name='property',
                            value_name='score'
                        )

        # Filter to only significant categories
        df_long = df_long.filter(pl.col('property').is_in(cats_to_show)).with_columns(
            pl.col('property').replace_strict({cats_to_show[i]: i for i in range(len(cats_to_show))}).alias('cat_order')
        ).sort(by='cat_order')
        
        # Format property names: split on "__" if present and wrap long text over multiple lines
        def format_property_name(prop_name, max_line_length=40):
            """Format property name to split IPR identifier from description and wrap long text."""
            def wrap_text(text, max_len):
                """Wrap text to multiple lines at word boundaries (underscores or spaces)."""
                if len(text) <= max_len:
                    return text
                lines = []
                current_line = ""
                # Split on underscores to preserve them, but also allow breaking at them
                # Replace underscores with a special marker, split, then restore
                parts = text.split('_')
                for i, part in enumerate(parts):
                    # Add underscore before part (except first)
                    separator = "_" if i > 0 else ""
                    test_line = current_line + separator + part if current_line else part
                    
                    if len(test_line) <= max_len:
                        current_line = test_line
                    else:
                        if current_line:
                            lines.append(current_line)
                        current_line = part
                if current_line:
                    lines.append(current_line)
                return "\n".join(lines) if len(lines) > 1 else text
            
            if '__' in prop_name:
                parts = prop_name.split('__', 1)
                ipr_part = parts[0]
                desc_part = parts[1]
                # Wrap the description part if it's long
                desc_wrapped = wrap_text(desc_part, max_line_length)
                return f"{ipr_part}\n{desc_wrapped}"
            else:
                # No "__" separator, just wrap the whole thing if it's long
                return wrap_text(prop_name, max_line_length)
        
        # Convert to pandas and format property names for plotting
        df_pd = df_long
        
        # Adjust number of columns based on number of categories
        cols = min(num_cols, len(cats_to_show))

        col_by = 'property'
        color_by = 'model_name'

        fig = sns.relplot(data = df_pd.with_columns(pl.col('model_name').replace_strict(model_rename_dict)), 
            x= xaxis_var, 
            y = 'score',
            col = col_by,
            col_wrap = cols,
            kind = 'line', 
            markers=True,
            errorbar = 'se',
            hue = color_by, 
            hue_order=[model_rename_dict.get(m) for m in model_order],
            palette = {model_rename_dict.get(k):v for k,v in model_colormapping.items()},
            height = 3,
            aspect = 1,
            facet_kws={'sharey': False, 'sharex': False},
            )

        fig.set_titles(col_template='{col_name}', row_template='{row_name}')

        for ax in fig.axes.flat:
            title_text = ax.get_title().replace('_', ' ')
           
            # Style for Residue-level: Bold and Blue
            ax.set_title(title_text, 
            color='black', 
            fontsize=10)
                

        leg = fig._legend
        leg.set_title('Model') # Your desired title string
        plt.setp(leg.get_title(), weight='bold')

        fig.set_axis_labels("layer #", "score")
        
        fig.tight_layout()
        fig.savefig(linear_probe_figures_dir / f"probing_{dataset}_{metric_to_plot}_special_categories_vs_{xaxis_var}_hue_{color_by}.png", dpi=300)

        plt.show()

    except Exception as e:
        print(e)
        continue

Picking categories that show differences between the models:

In [ ]:
# Configuration for plotting
plm_states = ['trained']
probe_states = ['trained']
model_order = ['amplify_120m', 'amplify_350m', 'samplify_120m', 'samplify_350m', 'esm2_150m', 'esm2_650m']
metric_to_plot = 'matthews_corrcoef_elementwise'  # Can be changed to other metrics
xaxis_var = 'layer_num'

# Helper function to extract model type and size
def get_model_info(model_name):
    parts = model_name.split('_')
    if 'esm2' in model_name:
        model_type = 'ESM2'
        size = int(parts[-1].replace('m', ''))
    elif 'amplify' in model_name:
        if 'samplify' in model_name: 
            model_type = 'SAMPLIFY'
        else:
            model_type = 'AMPLIFY'
        size = int(parts[-1].replace('m', ''))
    else:
        model_type = 'Other'
        size = 0
    return model_type, size

num_cols = 2

model_order = ['amplify_120m', 'amplify_350m', 'esm2_150m', 'esm2_650m']

cats_specific = {
    'interpro_conserved_site': [
        # 'IPR003903__ubiquitin_interacting_motif',
        'IPR029752__d-isomer_specific_2-hydroxyacid_dehydrogenase,_nad-binding_domain_conserved_site_1',
        'IPR020728__apoptosis_regulator,_bcl-2,_bh3_motif,_conserved_site',
    ],
    'interpro_domain': [
        'IPR000210__btb/poz_domain', 
        # 'IPR000488__death_domain', 
        # 'IPR000742__egf-like_domain', 
        'IPR003124__wh2_domain', 
        'IPR001909__krueppel-associated_box', 
        'IPR003309__scan_domain', 
        # 'IPR003599__immunoglobulin_domain_subtype',
    ],
    'interpro_family': [
        'IPR050427__human_olfactory_receptors', 
        'IPR050578__marvel_domain-containing_and_chemokine-like_factor_proteins',
    ]
}

# Plot all significant categories for each dataset
for dataset, cats_to_show in cats_specific.items():
    try:

        # Load original dataset (needed for label_cols)
        dataset_config = load_config(dataset)
        dataset_df = pl.read_parquet(dataset_config['data_path'])
        label_cols = [col for col in dataset_df.columns if (dataset_df[col].dtype != pl.String)]

        all_filters = [
            (pl.col('plm_embeddings_normalized') == False), # normalization filter
            (pl.col('plm_state').is_in(plm_states)), # plm training filter
            pl.col('linear_probe_state').is_in(probe_states), # probe training filter
            (pl.col('model_name').is_in(model_order)), # model filter
            (pl.col('dataset') == dataset), # dataset filter
            ~((pl.col('plm_state') == 'un-trained') & (pl.col('linear_probe_state') == 'un-trained')), # untrained filter
        ]

        # Apply filters
        combined_filter = reduce(lambda a, b: a & b, all_filters)

        # Remap names for legibility
        df_filtered = element_df.filter(combined_filter).with_columns(
                    pl.col('plm_state').replace_strict({'trained': 'trained PLM', 'un-trained': 'un-trained PLM'}), # renaming for label convenience
                    pl.col('linear_probe_state').replace_strict({'trained': 'trained probe', 'naive': 'naive probe', 'un-trained': 'un-trained probe'}),
                    pl.col('plm_embeddings_normalized').replace_strict({True: 'embeddings_normalized', False: 'embeddings_not_normalized'}),
                    pl.col('control_type').replace_strict({'original': 'original sequence', 'scrambled': 'scrambled sequence', 'mean': 'mean embedding', 'random_gaussian': 'gaussian_embedding', 'l2_normalized': 'normalized_embedding'}),
            ).with_columns(
                    pl.concat_str(['plm_state', 'linear_probe_state', 'control_type'], separator= ' & ').alias('condition'), # concatenate plm, linear probe and control type
            )

        # Add new columns for model type and size
        df_filtered = df_filtered.with_columns(
            # Use map_elements for Python function calls
            pl.col('model_name').map_elements(
                lambda x: get_model_info(x)[0], return_dtype=pl.String
            ).alias('model_type'),
            pl.col('model_name').map_elements(
                lambda x: get_model_info(x)[1], return_dtype=pl.Int64
            ).alias('model_size_m'),
        ).with_columns(
            (pl.col('model_type') + '_' + pl.col('model_size_m').cast(pl.String)).alias('model_group_size')
        )

        # Unpack the metric col
        cols_to_select = standard_cols + ['model_type', 'model_size_m', 'model_group_size']
        sub_df = df_filtered.select(cols_to_select + [metric_to_plot]).with_columns( # expand and rename columns
            pl.col(metric_to_plot).list.to_struct()
            ).unnest(metric_to_plot).rename(dict(zip([f'field_{i}' for i in range(len(label_cols))], label_cols)))        

        df_to_unpivot = sub_df.select(cols_to_select + label_cols)

        df_long = df_to_unpivot.unpivot(
                            index=cols_to_select,
                            on=label_cols,
                            variable_name='property',
                            value_name='score'
                        )

        # Filter to only significant categories
        df_long = df_long.filter(pl.col('property').is_in(cats_to_show)).with_columns(
            pl.col('property').replace_strict({cats_to_show[i]: i for i in range(len(cats_to_show))}).alias('cat_order')
        ).sort(by='cat_order')
        
        # Format property names: split on "__" if present and wrap long text over multiple lines
        def format_property_name(prop_name, max_line_length=40):
            """Format property name to split IPR identifier from description and wrap long text."""
            def wrap_text(text, max_len):
                """Wrap text to multiple lines at word boundaries (underscores or spaces)."""
                if len(text) <= max_len:
                    return text
                lines = []
                current_line = ""
                # Split on underscores to preserve them, but also allow breaking at them
                # Replace underscores with a special marker, split, then restore
                parts = text.split('_')
                for i, part in enumerate(parts):
                    # Add underscore before part (except first)
                    separator = "_" if i > 0 else ""
                    test_line = current_line + separator + part if current_line else part
                    
                    if len(test_line) <= max_len:
                        current_line = test_line
                    else:
                        if current_line:
                            lines.append(current_line)
                        current_line = part
                if current_line:
                    lines.append(current_line)
                return "\n".join(lines) if len(lines) > 1 else text
            
            if '__' in prop_name:
                parts = prop_name.split('__', 1)
                ipr_part = parts[0]
                desc_part = parts[1]
                # Wrap the description part if it's long
                desc_wrapped = wrap_text(desc_part, max_line_length)
                return f"{ipr_part}\n{desc_wrapped}"
            else:
                # No "__" separator, just wrap the whole thing if it's long
                return wrap_text(prop_name, max_line_length)
        
        # Convert to pandas and format property names for plotting
        df_pd = df_long.with_columns(pl.col('model_name').replace_strict(model_rename_dict)).to_pandas()
        df_pd['property_formatted'] = df_pd['property'].apply(format_property_name)
        
        # Adjust number of columns based on number of categories
        cols = min(num_cols, len(cats_to_show))

        col_by = 'property_formatted'
        color_by = 'model_name'

        fig = sns.relplot(data = df_pd, 
            x= xaxis_var, 
            y = 'score',
            col = col_by,
            col_wrap = cols,
            kind = 'line', 
            markers=True,
            errorbar = 'se',
            hue = color_by, 
            hue_order=[model_rename_dict.get(m) for m in model_order],
            palette = {model_rename_dict.get(k):v for k,v in model_colormapping.items()},
            height = 3,
            aspect = 1.25,
            facet_kws={'sharey': False, 'sharex': False},
            )

        fig.set_titles(col_template='{col_name}', row_template='{row_name}')

        fig.set_axis_labels("layer #", "score")

        # Figure Super Title
        fig.fig.suptitle(dataset.replace('_', ' ').title())

        leg = fig._legend
        leg.set_title('Model') # Your desired title string
        plt.setp(leg.get_title(), weight='bold')

        fig.tight_layout()
        
        fig.savefig(linear_probe_figures_dir / f"probing_{dataset}_{metric_to_plot}_special_categories_vs_{xaxis_var}_hue_{color_by}.png", dpi=300)
        print(f"Saved figure for {dataset}")

        plt.show()

    except Exception as e:
        print(e)
        continue
